# Shearing Noh Implosion (2D)

This notebook runs the shearing Noh implosion: the classical Noh converging-flow shock test with a transverse shear of amplitude `vs` superimposed on the initial velocity, so the converging shock has to survive a velocity discontinuity it is not aligned with. The `velocities` panel below plots the *x*-component (`mapping='x'` on `SHEARING_NOH_FIELDS`, not the magnitude) because the shear itself, not just its speed, is what this case is checking.

Like `08`/`09`/`10`, this is a `particlePlot` (2D field view) case: plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly on `SHEARING_NOH_FIELDS` (exported from `warpSPH.cases.shearingNoh`) rather than going through `shearingNohCase.setupPlot`/`updatePlot`, which do not live-update reliably inside a Jupyter cell in this environment.

`shearingNohCase` has no `timestep` hook, and `sampleShearingNoh` leaves `dt` unset, so this case needs an explicit `dt` (the case default, kept here) and the loop below is a fixed `range(nSteps)`.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/11-Shearing_Noh_2D.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.shearingNoh import shearingNohCase, SHEARING_NOH_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `11-shearing-noh-implosion-2d.py`, made explicit and editable here.
# `shearingNohCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=shearingNohCase.name, scheme=shearingNohCase.scheme,
                params=dict(shearingNohCase.params)) \
    .merged(**shearingNohCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=200,
    dim=2,
    L=2.0,

    # --- time stepping ---------------------------------------------------
    tLimit=0.6,
    # `sampleShearingNoh` leaves dt unset, and there is no `timestep` hook,
    # so this needs an explicit value that stays fixed -- the case default.
    dt=2.5e-4,

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- shearing Noh's own knob (shear amplitude) ----------------------------
    params=dict(
        vs=5.0,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`shearingNohCase.buildSystem` -> `sampleShearingNoh`), not re-derived here.
ctx = buildContext(shearingNohCase, spec)
shearingNohCase.configureScheme(ctx)
system = shearingNohCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(SHEARING_NOH_FIELDS), not shearingNohCase.setupPlot
# -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, SHEARING_NOH_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = shearingNohCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=shearingNohCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = shearingNohCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, SHEARING_NOH_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=shearingNohCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
